## XGBoost final model

Building the final model for the prediction, hyperparameter tuning and validation on the test data

In [1]:
import datetime as dt

import pandas as pd

from load_forecast.evaluation.backtest import build_forecasts, summarize
from load_forecast.evaluation.baselines import rte_day_ahead, seasonal_naive
from load_forecast.evaluation.data import (
    load_actuals,
    load_temperature,
    load_temperature_forecast,
)
from load_forecast.evaluation.protocol import TEST, TRAIN_END, VALIDATION, target_days
from load_forecast.model import (
    LEARNING_RATE,
    MAX_DEPTH,
    N_ESTIMATORS,
    TRAIN_START,
    build_training_data,
    make_forecast_fn,
    train_model,
)

TRAIN_END_WITH_VALIDATION = dt.date(2025, 8, 31)

In [2]:
actuals = load_actuals()
temperature = load_temperature()
temperature_forecast = load_temperature_forecast()
validation = target_days(*VALIDATION)

consumption = actuals["consumption_mw"]
X, y = build_training_data(actuals, temperature, TRAIN_END)
X = X.loc[TRAIN_START:]
y = y.loc[TRAIN_START:]

In [3]:
n_estimators_values = [100, 300, 600]
max_depth_values = [4, 6, 8]
learning_rate_values = [0.05, 0.1, 0.3]

In [4]:
rows = []
for n_estimators in n_estimators_values:
    for max_depth in max_depth_values:
        for learning_rate in learning_rate_values:
            model = train_model(
                X,
                y,
                n_estimators=n_estimators,
                max_depth=max_depth,
                learning_rate=learning_rate,
            )
            forecast = make_forecast_fn(model, temperature_forecast, consumption)
            forecasts = build_forecasts(forecast, consumption, validation)
            scores = summarize(forecasts)
            rows.append(
                {
                    "n_estimators": n_estimators,
                    "max_depth": max_depth,
                    "learning_rate": learning_rate,
                    "mae_mw": scores["mae_mw"].iloc[0],
                    "mape_pct": scores["mape_pct"].iloc[0],
                }
            )

grid = pd.DataFrame(rows).sort_values("mape_pct")
grid.round(2)

,n_estimators,max_depth,learning_rate,mae_mw,mape_pct
19,600,4,0.10,1216.02,2.35
20,600,4,0.30,1228.78,2.38
21,600,6,0.05,1232.33,2.38
11,300,4,0.30,1227.35,2.38
22,600,6,0.10,1235.38,2.38
13,300,6,0.10,1242.91,2.40
10,300,4,0.10,1250.97,2.43
18,600,4,0.05,1251.29,2.43
24,600,8,0.05,1259.50,2.43
15,300,8,0.05,1259.40,2.43


After testing the model with different hyperparameters, the best results on the validation set are achieved with n_estimators = 600, max_depth=4 and a learning rate of 0.10, achieving a MAPE of 2.35 %, against an MAPE of 2.47 % before hyperparameter tuning. Thisis the model we will keep forour forecasts

In [5]:
X_final, y_final = build_training_data(actuals, temperature, TRAIN_END_WITH_VALIDATION)
X_final = X_final.loc[TRAIN_START:]
y_final = y_final.loc[TRAIN_START:]

final_model = train_model(
    X_final,
    y_final,
    n_estimators=N_ESTIMATORS,
    max_depth=MAX_DEPTH,
    learning_rate=LEARNING_RATE,
)
final_forecast = make_forecast_fn(final_model, temperature_forecast, consumption)

In [6]:
def naive_7(slots):
    return seasonal_naive(consumption, slots, days=7)


def rte(slots):
    return rte_day_ahead(actuals["forecast_d1_mw"], slots)


test_methods = {
    "Naive 7 days": naive_7,
    "Final model": final_forecast,
    "RTE day-ahead": rte,
}

test_days = target_days(*TEST)
test_rows = []
for method_name, method in test_methods.items():
    forecasts = build_forecasts(method, consumption, test_days)
    scores = summarize(forecasts)
    scores["method_name"] = method_name
    scores["bias_mw"] = (forecasts["forecast_mw"] - forecasts["actual_mw"]).mean()
    test_rows.append(scores)

pd.concat(test_rows).set_index("method_name").round(2)

,mae_mw,mape_pct,bias_mw
method_name,,,
Naive 7 days,3445.45,6.42,-15.25
Final model,1247.11,2.41,-241.17
RTE day-ahead,1387.69,2.74,-1122.04


## Final results on the test period

The final model is XGBoost with calendar features, the consumption 2 and
7 days earlier, and the national temperature. It was trained from
January 2021 to August 2025, with 600 trees, a depth of 4 and a learning
rate of 0.1, chosen by grid search on the validation period. It is
evaluated with the temperature forecast made 48 hours ahead, so it only
uses information available at noon the day before.

| Method | MAE (MW) | MAPE (%) | Bias (MW) |
| --- | --- | --- | --- |
| Naive 7 days | 3,445 | 6.42 | -15 |
| Final model | 1,247 | 2.41 | -241 |
| RTE day-ahead | 1,388 | 2.74 | -1,122 |

The model beats the naive baseline by far and the RTE forecast by 0.33
percentage points. Its test MAPE is close to its validation MAPE
(2.35 %), so the choices made on the validation period hold on a new
year.